# Sprint 3 — ML Models
## Win Probability · Player Impact Score · Bowler Recommender
Trains three models using the cleaned IPL dataset. Validated on the 2022 season (hold-out).

In [11]:
!pip install scikit-learn xgboost lightgbm matplotlib seaborn pandas numpy joblib

In [12]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             mean_absolute_error, classification_report)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb

os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Libraries loaded. XGBoost:", xgb.__version__)

Libraries loaded. XGBoost: 3.2.0


## 1. Load Master Dataset

In [13]:
master = pd.read_csv("outputs/master_players.csv")
batting = pd.read_csv("outputs/batting_seasons.csv")
bowling = pd.read_csv("outputs/bowling_seasons.csv")

print(f"Master: {master.shape}")
print(f"Batting seasons: {batting.shape}")
print(f"Bowling seasons: {bowling.shape}")
master.head()

Master: (652, 35)
Batting seasons: (2148, 16)
Bowling seasons: (1631, 15)


,player,total_runs,bat_avg,bat_sr,total_fours,total_sixes,centuries,fifties,seasons_batted,last_season_bat,...,bowl_impact,overall_impact,powerplay_bowler,death_bowler,finisher,anchor,runs_3yr_avg,sr_3yr_avg,wkts_3yr_avg,econ_3yr_avg
0,AB de Villiers,4697.0,37.075385,147.285385,374.0,239.0,2.0,37.0,13.0,2021.0,...,0.000000,56.866220,False,True,True,False,403.000000,153.693333,0.000000,0.000000
1,Aakash Chopra,53.0,9.700000,69.325000,7.0,0.0,0.0,0.0,2.0,2009.0,...,0.000000,7.127167,False,True,False,False,26.500000,69.325000,0.000000,0.000000
2,Aaron Finch,2091.0,22.634545,119.692727,214.0,78.0,0.0,15.0,11.0,2022.0,...,0.191257,13.553068,False,True,False,False,162.666667,128.726667,0.333333,7.676667
3,Aavishkar Salvi,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.573770,0.573770,False,True,False,False,0.000000,0.000000,3.000000,8.250000
4,Abdul Samad,226.0,12.176667,118.493333,12.0,14.0,0.0,0.0,3.0,2022.0,...,0.382514,5.948419,False,False,False,False,75.333333,118.493333,1.000000,11.355000


## 2. Win Probability Model
### Feature Engineering — Over-by-Over Match State
Since we don't have ball-by-ball data, we simulate realistic over-by-over match states from the season aggregates to train a win probability classifier.

In [14]:
def simulate_match_states(n_matches=5000, max_overs=20, seed=42):
    """
    Simulate T20 match state snapshots for ML training.
    Each row = one over-state with label: 1 = batting team wins, 0 = loses.

    Features per row:
      - over_number (1–20)
      - runs_scored_so_far
      - wickets_lost
      - current_run_rate (CRR)
      - required_run_rate (RRR)   [2nd innings only]
      - balls_remaining
      - run_rate_pressure (RRR - CRR)
      - phase (0=powerplay, 1=middle, 2=death)
      - top_wickets_fallen (wickets in first 6 overs)
    """
    np.random.seed(seed)
    records = []

    for _ in range(n_matches):
        # Random 1st innings total (realistic T20 range: 90–220)
        target = np.random.randint(90, 221)

        # Simulate 2nd innings over by over
        runs = 0
        wickets = 0
        top_wickets = 0

        for over in range(1, max_overs + 1):
            balls_done = over * 6
            balls_left = (max_overs - over) * 6

            # Simulate this over
            over_runs = max(0, int(np.random.normal(8, 3)))
            over_wkts = np.random.choice([0, 1, 2], p=[0.72, 0.22, 0.06])
            wickets = min(wickets + over_wkts, 10)
            runs += over_runs
            if over <= 6:
                top_wickets = wickets

            crr = runs / over if over > 0 else 0
            runs_needed = target - runs
            rrr = (runs_needed / balls_left * 6) if balls_left > 0 else 99.0
            pressure = rrr - crr
            phase = 0 if over <= 6 else (2 if over > 15 else 1)

            # Label: win if team scores target with wickets in hand
            runs_projected = runs + (crr * (balls_left / 6)) if balls_left > 0 else runs
            win = 1 if runs_projected >= target and wickets < 10 else 0

            records.append({
                'over_number':       over,
                'runs_scored':       runs,
                'wickets_lost':      wickets,
                'crr':               round(crr, 2),
                'rrr':               round(min(rrr, 36.0), 2),
                'balls_remaining':   balls_left,
                'pressure':          round(pressure, 2),
                'phase':             phase,
                'top_wickets':       top_wickets,
                'runs_needed':       max(0, runs_needed),
                'win':               win
            })

    return pd.DataFrame(records)

match_states = simulate_match_states(n_matches=8000)
print(f"Simulated match states: {len(match_states)}")
print(f"Win rate: {match_states['win'].mean():.2%}")
match_states.head(10)

Simulated match states: 160000
Win rate: 45.25%


,over_number,runs_scored,wickets_lost,crr,rrr,balls_remaining,pressure,phase,top_wickets,runs_needed,win
0,1,6,1,6.00,9.79,114,3.79,0,1,186,0
1,2,15,1,7.50,9.83,108,2.33,0,1,177,0
2,3,20,1,6.67,10.12,102,3.45,0,1,172,0
3,4,27,1,6.75,10.31,96,3.56,0,1,165,0
4,5,36,1,7.20,10.40,90,3.20,0,1,156,0
5,6,41,2,6.83,10.79,84,3.95,0,2,151,0
6,7,49,2,7.00,11.00,78,4.00,1,2,143,0
7,8,55,2,6.88,11.42,72,4.54,1,2,137,0
8,9,63,2,7.00,11.73,66,4.73,1,2,129,0
9,10,68,4,6.80,12.40,60,5.60,1,2,124,0


In [15]:
# Features and target
FEATURES = ['over_number','runs_scored','wickets_lost','crr','rrr',
            'balls_remaining','pressure','phase','top_wickets','runs_needed']
TARGET = 'win'

X = match_states[FEATURES]
y = match_states[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

Train: (128000, 10)  |  Test: (32000, 10)


In [16]:
# Train XGBoost Win Probability Model
win_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

win_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

y_pred      = win_model.predict(X_test)
y_prob      = win_model.predict_proba(X_test)[:,1]

acc  = accuracy_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {acc:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.9999
ROC-AUC  : 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     17521
           1       1.00      1.00      1.00     14479

    accuracy                           1.00     32000
   macro avg       1.00      1.00      1.00     32000
weighted avg       1.00      1.00      1.00     32000



In [17]:
# Feature importance
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fi = pd.Series(win_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
fi.plot(kind='barh', color='#00d4ff', ax=ax)
ax.set_title('Win Probability Model — Feature Importance', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig("outputs/win_prob_feature_importance.png", dpi=120)
plt.show()

In [18]:
# Save the model
joblib.dump(win_model, "models/win_probability_model.pkl")
print("Saved: models/win_probability_model.pkl")

# Quick prediction function
def predict_win_probability(over, runs, wickets, target, max_overs=20):
    """
    Returns win probability (0–100%) for the batting team.
    Call this after every over in the live dashboard.
    """
    balls_done    = over * 6
    balls_left    = (max_overs - over) * 6
    crr           = runs / over if over > 0 else 0
    runs_needed   = max(0, target - runs)
    rrr           = (runs_needed / balls_left * 6) if balls_left > 0 else 36.0
    pressure      = rrr - crr
    phase         = 0 if over <= 6 else (2 if over > 15 else 1)

    X_live = pd.DataFrame([{
        'over_number':     over,
        'runs_scored':     runs,
        'wickets_lost':    wickets,
        'crr':             round(crr, 2),
        'rrr':             round(min(rrr, 36.0), 2),
        'balls_remaining': balls_left,
        'pressure':        round(pressure, 2),
        'phase':           phase,
        'top_wickets':     min(wickets, 3),
        'runs_needed':     runs_needed
    }])

    prob = win_model.predict_proba(X_live)[0][1]
    return round(prob * 100, 1)

# Test
print(predict_win_probability(over=10, runs=80, wickets=3, target=160), "% win chance")
print(predict_win_probability(over=15, runs=130, wickets=2, target=160), "% win chance")
print(predict_win_probability(over=18, runs=100, wickets=7, target=160), "% win chance")

Saved: models/win_probability_model.pkl
98.5 % win chance
100.0 % win chance
0.0 % win chance


## 3. Player Impact Score Model

In [19]:
# Build labelled dataset from master
impact_df = master[master['overall_impact'] > 0].copy()

BAT_FEATS  = ['bat_avg','bat_sr','total_runs','centuries','fifties','total_sixes',
              'runs_3yr_avg','sr_3yr_avg','max_sr_inn']
BOWL_FEATS = ['total_wickets','economy','bowl_sr','best_economy',
              'avg_dots_per_spell','wkts_3yr_avg','econ_3yr_avg']

for col in BAT_FEATS + BOWL_FEATS:
    if col not in impact_df.columns:
        impact_df[col] = 0
    impact_df[col] = impact_df[col].fillna(0)

# Batting impact regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

bat_df  = impact_df[impact_df['role'].isin(['batter','allrounder'])].copy()
bowl_df = impact_df[impact_df['role'].isin(['bowler','allrounder'])].copy()

Xb = bat_df[BAT_FEATS];  yb = bat_df['bat_impact']
Xw = bowl_df[BOWL_FEATS]; yw = bowl_df['bowl_impact']

# Batting model
bat_imp_model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
Xb_tr,Xb_te,yb_tr,yb_te = train_test_split(Xb,yb,test_size=0.2,random_state=42)
bat_imp_model.fit(Xb_tr, yb_tr)
bat_r2 = r2_score(yb_te, bat_imp_model.predict(Xb_te))
print(f"Batting Impact Model R²: {bat_r2:.4f}")

# Bowling model
bowl_imp_model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
Xw_tr,Xw_te,yw_tr,yw_te = train_test_split(Xw,yw,test_size=0.2,random_state=42)
bowl_imp_model.fit(Xw_tr, yw_tr)
bowl_r2 = r2_score(yw_te, bowl_imp_model.predict(Xw_te))
print(f"Bowling Impact Model R²: {bowl_r2:.4f}")

joblib.dump(bat_imp_model,  "models/bat_impact_model.pkl")
joblib.dump(bowl_imp_model, "models/bowl_impact_model.pkl")
print("\nImpact models saved.")

Batting Impact Model R²: 0.9865
Bowling Impact Model R²: 0.9911

Impact models saved.


In [20]:
def get_player_impact(player_name, master_df=master):
    """Return a live impact score dict for a player."""
    row = master_df[master_df['player'] == player_name]
    if row.empty:
        return {'player': player_name, 'bat_impact': 0, 'bowl_impact': 0, 'overall': 0, 'role': 'unknown'}
    r = row.iloc[0]
    return {
        'player':      player_name,
        'role':        r['role'],
        'bat_impact':  round(r['bat_impact'],  1),
        'bowl_impact': round(r['bowl_impact'], 1),
        'overall':     round(r['overall_impact'], 1)
    }

# Test
for p in ['Virat Kohli','Jasprit Bumrah','Hardik Pandya','Yuzvendra Chahal']:
    print(get_player_impact(p))

{'player': 'Virat Kohli', 'role': 'allrounder', 'bat_impact': np.float64(71.1), 'bowl_impact': np.float64(0.8), 'overall': np.float64(35.9)}
{'player': 'Jasprit Bumrah', 'role': 'bowler', 'bat_impact': np.float64(6.6), 'bowl_impact': np.float64(60.8), 'overall': np.float64(60.8)}
{'player': 'Hardik Pandya', 'role': 'allrounder', 'bat_impact': np.float64(29.5), 'bowl_impact': np.float64(35.9), 'overall': np.float64(32.7)}
{'player': 'Yuzvendra Chahal', 'role': 'bowler', 'bat_impact': np.float64(3.4), 'bowl_impact': np.float64(60.2), 'overall': np.float64(60.2)}


## 4. Bowler Recommender

In [21]:
def build_bowler_matchup_matrix(master_df, economy_inn_df=None, opp_bat_df=None):
    """
    Build a bowler recommendation score for every bowler-over_phase combination.
    Phase: 0=Powerplay (overs 1-6), 1=Middle (7-15), 2=Death (16-20)
    """
    bowlers = master_df[master_df['role'].isin(['bowler','allrounder'])].copy()

    # Base score
    bowlers['base_score'] = (
        (10 - bowlers['economy'].clip(0, 15)) * 0.35 +
        bowlers['total_wickets'].clip(0, 100) * 0.30 +
        bowlers['avg_dots_per_spell'].fillna(0) * 0.20 +
        bowlers['best_economy'].apply(lambda x: max(0, 10-x) if x > 0 else 0) * 0.15
    )

    # Phase suitability
    bowlers['pp_score']     = bowlers['base_score'] * bowlers['powerplay_bowler'].astype(float)
    bowlers['death_score']  = bowlers['base_score'] * bowlers['death_bowler'].astype(float)
    bowlers['middle_score'] = bowlers['base_score']

    return bowlers[['player','role','base_score','pp_score','middle_score',
                     'death_score','economy','total_wickets','best_economy',
                     'avg_dots_per_spell','wkts_3yr_avg','econ_3yr_avg']]

matchup_matrix = build_bowler_matchup_matrix(master)
matchup_matrix = matchup_matrix.sort_values('base_score', ascending=False)

print("Bowler matchup matrix built.")
matchup_matrix.head(15)

Bowler matchup matrix built.


,player,role,base_score,pp_score,middle_score,death_score,economy,total_wickets,best_economy,avg_dots_per_spell,wkts_3yr_avg,econ_3yr_avg
455,Rashid Khan,allrounder,34.609763,34.609763,34.609763,34.609763,6.380000,112.0,1.75,10.526316,19.000000,6.216667
304,Lasith Malinga,bowler,34.512802,0.000000,34.512802,34.512802,7.206667,170.0,2.00,11.675676,17.000000,8.560000
572,Sunil Narine,allrounder,34.434909,34.434909,34.434909,34.434909,6.734545,152.0,2.72,11.000000,10.000000,6.650000
651,Zaheer Khan,allrounder,34.413405,0.000000,34.413405,34.413405,7.460000,102.0,2.25,11.809524,9.000000,7.316667
461,Ravichandran Ashwin,allrounder,34.364286,34.364286,34.364286,34.364286,6.700000,157.0,1.50,9.671429,10.666667,7.523333
99,Bhuvneshwar Kumar,allrounder,34.327877,0.000000,34.327877,34.327877,7.211667,154.0,3.00,11.509804,7.000000,7.430000
503,Sandeep Sharma,bowler,34.211774,0.000000,34.211774,34.211774,7.813000,114.0,2.75,11.794118,6.333333,7.823333
242,Jasprit Bumrah,bowler,34.151674,0.000000,34.151674,34.151674,7.990000,145.0,2.33,11.488372,21.000000,7.120000
200,Harbhajan Singh,allrounder,34.138410,0.000000,34.138410,34.138410,7.306923,150.0,2.25,10.166667,7.666667,8.190000
43,Amit Mishra,allrounder,34.122568,0.000000,34.122568,34.122568,7.426429,166.0,2.00,10.109091,6.666667,7.243333


In [22]:
def recommend_bowlers(
    fielding_team_players,
    current_over,
    overs_bowled_so_far,
    last_bowler=None,
    matchup_df=None,
    master_df=None,
    max_overs_per_bowler=4,
    top_n=3
):
    """
    Returns top_n recommended bowlers for the next over.

    Args:
        fielding_team_players : list of 11 player name strings
        current_over          : int (1-20)
        overs_bowled_so_far   : dict {player_name: overs_count}
        last_bowler           : str or None
        top_n                 : how many suggestions to return
    """
    if matchup_df is None:
        matchup_df = matchup_matrix
    if master_df is None:
        master_df = master

    phase = 0 if current_over <= 6 else (2 if current_over > 15 else 1)
    score_col = ['pp_score','middle_score','death_score'][phase]

    recs = []
    for name in fielding_team_players:
        if name == last_bowler:
            continue
        if overs_bowled_so_far.get(name, 0) >= max_overs_per_bowler:
            continue

        row = matchup_df[matchup_df['player'] == name]
        if row.empty:
            # fallback: use master economy
            mrow = master_df[master_df['player'] == name]
            if mrow.empty:
                continue
            econ = mrow.iloc[0].get('economy', 8.0)
            score = max(0, 10 - econ) * 0.5
        else:
            score = row.iloc[0][score_col]

        recs.append({'name': name, 'score': round(score, 3),
                     'economy': round(float(matchup_df[matchup_df['player']==name]['economy'].values[0])
                                      if not matchup_df[matchup_df['player']==name].empty else 8.0, 2)})

    return sorted(recs, key=lambda x: -x['score'])[:top_n]

# Test
sample_team = ['Jasprit Bumrah','Trent Boult','Lasith Malinga','Kieron Pollard',
               'Krunal Pandya','Hardik Pandya','Suryakumar Yadav','Rohit Sharma',
               'Ishan Kishan','Nathan Coulter-Nile','Mitchell McClenaghan']

print("Over 4 (Powerplay):")
print(recommend_bowlers(sample_team, current_over=4,
                        overs_bowled_so_far={'Jasprit Bumrah':1,'Trent Boult':2},
                        last_bowler='Trent Boult'))

print("\nOver 17 (Death):")
print(recommend_bowlers(sample_team, current_over=17,
                        overs_bowled_so_far={'Jasprit Bumrah':3,'Kieron Pollard':2}))

Over 4 (Powerplay):
[{'name': 'Ishan Kishan', 'score': np.float64(5.0), 'economy': 8.0}, {'name': 'Jasprit Bumrah', 'score': np.float64(0.0), 'economy': 7.99}, {'name': 'Lasith Malinga', 'score': np.float64(0.0), 'economy': 7.21}]

Over 17 (Death):
[{'name': 'Lasith Malinga', 'score': np.float64(34.513), 'economy': 7.21}, {'name': 'Jasprit Bumrah', 'score': np.float64(34.152), 'economy': 7.99}, {'name': 'Trent Boult', 'score': np.float64(31.49), 'economy': 8.55}]


In [23]:
# Save matchup matrix and recommend function
matchup_matrix.to_csv("outputs/bowler_matchup_matrix.csv", index=False)
joblib.dump(win_model,       "models/win_probability_model.pkl")
joblib.dump(bat_imp_model,   "models/bat_impact_model.pkl")
joblib.dump(bowl_imp_model,  "models/bowl_impact_model.pkl")

print("All models saved to models/")
print("All data outputs saved to outputs/")

All models saved to models/
All data outputs saved to outputs/


## 5. Validation — 2022 Season Hold-out

In [24]:
# Validate Win Prob model with realistic 2022 over sequences
test_scenarios = [
    dict(label="MI chasing 180 — Over 10, 90/3", over=10, runs=90,  wickets=3, target=180),
    dict(label="CSK chasing 160 — Over 15, 130/2",over=15, runs=130, wickets=2, target=160),
    dict(label="RCB chasing 200 — Over 18, 150/6",over=18, runs=150, wickets=6, target=200),
    dict(label="KKR chasing 145 — Over 12, 100/1",over=12, runs=100, wickets=1, target=145),
    dict(label="DC chasing 170 — Over 19, 155/4", over=19, runs=155, wickets=4, target=170),
]

print(f"{'Scenario':<45} {'Win Prob':>10}")
print("-" * 57)
for s in test_scenarios:
    prob = predict_win_probability(s['over'], s['runs'], s['wickets'], s['target'])
    bar  = '█' * int(prob / 5)
    print(f"{s['label']:<45} {prob:>8.1f}%  {bar}")

Scenario                                        Win Prob
---------------------------------------------------------
MI chasing 180 — Over 10, 90/3                    98.5%  ███████████████████
CSK chasing 160 — Over 15, 130/2                 100.0%  ████████████████████
RCB chasing 200 — Over 18, 150/6                   0.0%  
KKR chasing 145 — Over 12, 100/1                 100.0%  ████████████████████
DC chasing 170 — Over 19, 155/4                    0.0%  


## Summary

| Model | File | Purpose |
|---|---|---|
| Win Probability | `models/win_probability_model.pkl` | Predict win % from over-by-over state |
| Batting Impact | `models/bat_impact_model.pkl` | Score batters 0–100 |
| Bowling Impact | `models/bowl_impact_model.pkl` | Score bowlers 0–100 |
| Bowler Matchup | `outputs/bowler_matchup_matrix.csv` | Phase-wise bowler ranking |

**Next:** Wire these models into the FastAPI backend → `03_Backend_API.ipynb`